In [12]:
from langgraph.graph import StateGraph,START,END
from langgraph.graph import add_messages
from langchain_core.messages import AIMessage,SystemMessage,BaseMessage
from typing import TypedDict,Annotated,List
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode,tools_condition
from dotenv import load_dotenv
from langgraph.types import interrupt,Command
import os
import requests

In [2]:
load_dotenv(dotenv_path='.env', override=True)

True

In [3]:

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [7]:


llm = ChatGroq(
    model="llama-3.3-70b-versatile",  
    api_key=GROQ_API_KEY  
)

In [9]:
@tool
def get_stock_price(symbol: str) -> dict:
    """
    Fetch latest stock price for a given symbol (e.g. 'AAPL', 'TSLA') 
    using Alpha Vantage with API key in the URL.
    """
    url = (
        "https://www.alphavantage.co/query"
        f"?function=GLOBAL_QUOTE&symbol={symbol}&apikey=C9PE94QUEW9VWGFM"
    )
    r = requests.get(url)
    return r.json()

In [10]:
@tool
def purchase_stock(symbol:str,quantity:float) ->dict:
    """
    Simulate purchasing a given quantity of a stock symbol.

    HUMAN-IN-THE-LOOP:
    Before confirming the purchase, this tool will interrupt
    and wait for a human decision ("yes" / anything else).
    """
    # This pauses the graph and returns control to the caller
    decision = interrupt(f"Approve buying {quantity} shares of {symbol}? (yes/no)")
    if isinstance(decision, str) and decision.lower() == "yes":
        return{
            "status": "success",
            "message": f"Purchase order placed for {quantity} shares of {symbol}.",
            "symbol": symbol,
            "quantity": quantity,
        }
    else:
        
        return {
            "status": "cancelled",
            "message": f"Purchase of {quantity} shares of {symbol} was declined by human.",
            "symbol": symbol,
            "quantity": quantity,
        }


In [11]:
tools=[get_stock_price,purchase_stock]
llm_with_tools=llm.bind_tools(tools)

In [14]:
class chatstate(TypedDict):
    messages:Annotated[list[BaseMessage],add_messages]

In [19]:
graph=StateGraph(chatstate)


In [20]:
def chat_node(state:chatstate):
    message=state['messages']
    response=llm_with_tools.invoke(message)
    return{'messages':[response]}
tool_node=ToolNode(tools)

In [21]:
graph.add_node('chat_node',chat_node)
graph.add_node('tools',tools)
graph.add_edge(START,'chat_node')
graph.add_conditional_edges('chat_node',tools_condition)
graph.add_edge('tools','chat_node')
graph.add_edge('chat_node',END)

TypeError: Expected a Runnable, callable or dict.Instead got an unsupported type: <class 'list'>